In [ ]:
# ============================================================================
# CONFIGURAZIONE INIZIALE E DEFINIZIONE DEI SIMBOLI
# ============================================================================
# Questo notebook calcola l'ampiezza per il decadimento B → K ℓℓ
# utilizzando il formalismo delle ampiezze di elicità

# --- Import delle librerie necessarie ---
from tqdm.auto import tqdm  # Barra di progresso
import sympy as sp  # Calcolo simbolico
from sympy.physics.quantum.spin import Rotation as rot  # Matrici di rotazione di Wigner
from sympy import pi
from random import randint

# Configurazione per la stampa formattata delle espressioni
sp.init_printing(use_unicode=True, wrap_line=True, num_columns=120)

# --- Parametri di troncamento ---
# Momento angolare massimo
Jmax = 2
Lmax = 2

# --- Definizione dei simboli per i momenti angolari ---
# J: spin della risonanza intermedia
# LB, LR, L1, L2: momenti angolari orbitali
# JX: spin della risonanza fittizia
# lam, lam1, lam2: elicità delle particelle
# eta: chiralità della corrente leptonica (±1)
J, LB, LR, L1, L2, JX, lam, lam1, lam2, eta = sp.symbols('J LB LR L1 L2 JX lam lam1 lam2 eta', integer=True)

# --- Variabili angolari ---
theta1, theta2, thetal, thetaK, phi, chi = sp.symbols('theta1 theta2 thetal thetaK phi chi', real=True)
eta_par = sp.symbols('eta_par', integer=True)  # chiralità per le ampiezze di elicità leptoniche
Res = sp.symbols('Res')  # Etichetta per la risonanza

# --- Funzioni simboliche per le ampiezze di elicità ---
# H: Ampiezza di elicità per la transizione B → K_J
H = sp.Function('H', complex=True)(eta, J, lam, LB)
# Hbar: Ampiezza di elicità per la transizione B → K*_J
Hbar = sp.Function('Hbar', complex=True)(eta, J, lam, LB)
# LHA: Ampiezza di elicità leptonica
LHA = sp.Function('LHA', real=True)(eta_par, lam1, lam2)
# G: Ampiezza di elicità effettiva per la transizione K_J → R P1
G = sp.Function('G', complex=True)(eta, J, lam, LB)

# --- Propagatore di Breit-Wigner ---
Prop = sp.Function('R')(Res)

# --- Fattori di forma di Blatt-Weisskopf ---
L, rBW = sp.symbols('L rBW', positive=True, real=True)  # L: momento angolare, rBW: raggio
rB, r3, r2 = sp.symbols('rB r3 r2', positive=True, real=True)  # Raggi di interazione
q = sp.symbols('q', real=True)  # Break-up momentum
qR = sp.symbols('qR', real=True)  # Break-up momentum alla risonanza

# Funzione simbolica generica per il fattore di Blatt-Weisskopf
BWfunc = sp.Function('BW', real=True)
BW = BWfunc(L, rBW, qR)  # Istanza base con argomenti simbolici

# --- Lista delle ampiezze totali ---
# Memorizza tutti i contributi all'ampiezza (risonanze + non-risonante)
atot = []
a_NR = sp.symbols('a_NR', complex=True)  # Coefficiente per il contributo non-risonante
atot.append(a_NR)  # Aggiunge il contributo non-risonante alla lista


In [ ]:
# =============================
# COSTRUZIONE DELL'AMPIEZZA 
# =============================

# --- Inizializzazione dell'ampiezza ---
AR = 0  # Ampiezza totale
J_val = 0  # Valore dello spin del K finale

# --- Definizione delle elicità dei leptoni ---
lambda1 = sp.symbols('lambda1', real=True)  # Elicità del leptone carico negativo
lambda2 = sp.symbols('lambda2', real=True)  # Elicità del leptone carico positivo
lam_val = 0  # Elicità del K

Hterm = H.subs({J: J_val, lam: lam_val, LB: 1})
term_vector = (
    -Hterm * LHA.subs(eta_par, eta).subs(lam1, lambda1).subs(lam2, lambda2) *
    rot.d(1, lam_val, lambda1 - lambda2, thetal)
)
AR += term_vector

# --- Contributo timelike ---
term_timelike = 0
t = sp.symbols('t', real=True)
Ht = sp.symbols('Ht', complex=True)                  # Ampiezza di elicità adronica timelike
Lt = sp.Function('Lt', complex=True)(lam1, lam2)     # Ampiezza di elicità leptonica timelike
term_timelike = (
    Ht * Lt.subs(lam1, lambda1).subs(lam2, lambda2)
)

AR += term_timelike

# Mostra l'ampiezza totale
print("\n" + "="*50)
if AR == 0:
    print("L'ampiezza AR è nulla (nessuna risonanza selezionata).")
else:
    print("Ampiezza AR totale:")
    print(f"Termini nell'ampiezza AR: {len(AR.as_ordered_terms())}")
print("="*50)

sp.pprint(AR)


Ampiezza AR totale:
Termini nell'ampiezza AR: 2
                                              1                
Ht⋅Lt(λ₁, λ₂) - H(η, 0, 0, 1)⋅LHA(η, λ₁, λ₂)⋅d         (thetal)
                                              0,λ₁ - λ₂        


In [ ]:
# =============================
# CALCOLO DEL QUADRATO DELL'AMPIEZZA TOTALE
# =============================

# Ampiezze di elicità leptoniche
b = sp.symbols('b', real = True) # beta = sqrt(1 - 4 m_ell^2/q^2)
ml = sp.symbols('ml', real = True) # massa dei leptoni

Leptonic_subs = {
    LHA.subs({eta_par: -1, lam1: sp.Rational(-1,2), lam2: sp.Rational(1,2)}): (1+b)*sp.sqrt(2)*q,
    LHA.subs({eta_par: -1, lam1: sp.Rational(1,2), lam2: sp.Rational(-1,2)}): (1-b)*sp.sqrt(2)*q,
    LHA.subs({eta_par: -1, lam1: sp.Rational(-1,2), lam2: sp.Rational(-1,2)}): 2*ml,
    LHA.subs({eta_par: -1, lam1: sp.Rational(1,2), lam2: sp.Rational(1,2)}): 2*ml,

    LHA.subs({eta_par: 1, lam1: sp.Rational(-1,2), lam2: sp.Rational(1,2)}): (1-b)*sp.sqrt(2)*q,
    LHA.subs({eta_par: 1, lam1: sp.Rational(1,2), lam2: sp.Rational(-1,2)}): (1+b)*sp.sqrt(2)*q,
    LHA.subs({eta_par: 1, lam1: sp.Rational(-1,2), lam2: sp.Rational(-1,2)}): 2*ml,
    LHA.subs({eta_par: 1, lam1: sp.Rational(1,2), lam2: sp.Rational(1,2)}): 2*ml,

    Lt.subs({lam1: sp.Rational(-1,2), lam2: sp.Rational(-1,2)}): 2*ml,
    Lt.subs({lam1: sp.Rational(1,2), lam2: sp.Rational(1,2)}): -2*ml,
    Lt.subs({lam1: sp.Rational(-1,2), lam2: sp.Rational(1,2)}): 0,
    Lt.subs({lam1: sp.Rational(1,2), lam2: sp.Rational(-1,2)}): 0,
}

# MODULO QUADRO AMPIEZZA TOTALE
Atot = AR
Atotsqfin = 0

for lambda1_val in [sp.Rational(-1,2), sp.Rational(1,2)]:
    for lambda2_val in [sp.Rational(-1,2), sp.Rational(1,2)]:
        
        Term_L = Atot.subs({eta: -1, lambda1: lambda1_val, lambda2: lambda2_val})
        Term_R = Atot.subs({eta: 1, lambda1: lambda1_val, lambda2: lambda2_val})
        
        Amp_Total_Helicity = Term_L + Term_R
        Amp_Total_Helicity = Amp_Total_Helicity.doit().subs(Leptonic_subs)
        
        term_sq = Amp_Total_Helicity * sp.conjugate(Amp_Total_Helicity)
        
        Atotsqfin += term_sq
N = (8) # Fattore di normalizzazione costante
Atotsqfin = sp.expand((Atotsqfin)/N)

terms_list = Atotsqfin.as_ordered_terms()
print(f"Numero di termini: {len(terms_list)}\n")
print(f"Numero di parametri a_i: {len(atot)}\n")

sp.pprint(Atotsqfin)

Numero di termini: 13

Numero di parametri a_i: 1

               2  2                   2         ______________    2  2                   2         _____________    2  ↪
       2 __   b ⋅q ⋅H(-1, 0, 0, 1)⋅sin (thetal)⋅H(-1, 0, 0, 1)   b ⋅q ⋅H(-1, 0, 0, 1)⋅sin (thetal)⋅H(1, 0, 0, 1)   b ⋅ ↪
4⋅Ht⋅ml ⋅Ht + ──────────────────────────────────────────────── - ─────────────────────────────────────────────── - ─── ↪
                                     4                                                  4                              ↪

↪  2                  2         ______________    2  2                  2         _____________                        ↪
↪ q ⋅H(1, 0, 0, 1)⋅sin (thetal)⋅H(-1, 0, 0, 1)   b ⋅q ⋅H(1, 0, 0, 1)⋅sin (thetal)⋅H(1, 0, 0, 1)     2                  ↪
↪ ──────────────────────────────────────────── + ────────────────────────────────────────────── + ml ⋅H(-1, 0, 0, 1)⋅c ↪
↪                     4                                                4                             

In [ ]:
# =============================
# Ampiezze di elicità in termini dei fattori di forma associati alle struture di Lorentz
# =============================

mB, mK = sp.symbols('mB mK', real=True, positive=True) #masse dei mesoni B e K
FV,FA,FP = sp.symbols('FV FA FP', complex=True) #Fattori di forma associati alle strutture di Lorentz: Vettoriale, Assiale, Pseudoscalare
lam_kin = sp.Symbol("lambda", real=True, positive=True) #funzione di Kallen
norm_long = sp.sqrt(lam_kin) / sp.sqrt(q**2) 
scalar_kin_factor = (mB**2 - mK**2 + q**2) / sp.sqrt(q**2)
half = sp.Rational(1,2)

helicity_to_transverity_subs = {
    H.subs({eta: -1, J: 0, lam: 0, LB: 1}): half * norm_long*(FV - FA),
    H.subs({eta: 1, J: 0, lam: 0, LB: 1}): half * norm_long*(FV + FA),
    Ht: half*((sp.sqrt(q**2)/(ml)) * FP + scalar_kin_factor * FA),
    sp.sin(thetal)**2: 1-sp.cos(thetal)**2,
}
Atotsq_mod = Atotsqfin.subs(helicity_to_transverity_subs).expand()

In [ ]:
# =============================
# Funzione per l'estrazione della struttura trigonometrica dai termini del modulo quadro dell'ampiezza totale
# =============================

def extract_trig_structure(expr):
    """Estrae la struttura trigonometrica completa da un'espressione"""
    trig_factors = []
    non_trig_factors = []
    
    # Scompone il termine in fattori
    if expr.is_Mul: #controlla se è un prodotto 'Mul' per Sympy e li ordina come termini
        factors = expr.as_ordered_factors()
    elif expr.is_Pow:
        # Gestisci le potenze
        base, exp = expr.as_base_exp() # Divide in base ed esponente
        if base.atoms(sp.sin, sp.cos):
            return [expr], []  # Tutta la potenza è trigonometrica
        else:
            return [], [expr]  # Tutta la potenza è non-trigonometrica
    else:
        factors = [expr]
    
    for factor in factors:
        if factor.atoms(sp.sin, sp.cos):
            # Questo fattore contiene funzioni trigonometriche
            if factor.is_Pow:
                # È una potenza di funzione trigonometrica
                trig_factors.append(factor)
            elif factor.is_Mul:
                # È un prodotto che contiene trigonometria
                sub_trig, sub_non_trig = extract_trig_structure(factor)
                trig_factors.extend(sub_trig) #extend aggiunge in coda tutti gli elementi della lista 'sub_trig'
                non_trig_factors.extend(sub_non_trig)
            else:
                # È una singola funzione trigonometrica
                trig_factors.append(factor)
        else:
            non_trig_factors.append(factor)
    
    return trig_factors, non_trig_factors

# Raccoglie tutti i prodotti trigonometrici unici
trig_coefficients = {}

print("Analizzando ogni termine dell'espressione...")
for i, term in tqdm(enumerate(Atotsq_mod.as_ordered_terms()), total=len(Atotsq_mod.as_ordered_terms()), desc="Analisi termini angolari"):
    
    # Estrai la struttura trigonometrica
    trig_factors, non_trig_factors = extract_trig_structure(term)
    
    # Costruisci il prodotto trigonometrico
    if trig_factors:
        trig_product = 1
        for tf in trig_factors:
            trig_product *= tf
        trig_key = str(trig_product)
    else:
        trig_key = "COSTANTE"
    
    # Costruisci il coefficiente non trigonometrico
    if non_trig_factors:
        non_trig_product = 1
        for ntf in non_trig_factors:
            non_trig_product *= ntf
    else:
        non_trig_product = 1
    
    # Accumula i coefficienti
    if trig_key in trig_coefficients:
        trig_coefficients[trig_key] += non_trig_product
    else:
        trig_coefficients[trig_key] = non_trig_product

print(f"\nTrovati {len(trig_coefficients)} prodotti trigonometrici distinti")

# Ordina per lunghezza della stringa (prima i più semplici)
print("Ordinando i prodotti trigonometrici per complessità")
sorted_keys = sorted(trig_coefficients.keys(), key=lambda x: (len(x), x))

print(f"\n" + "="*70)
print("RIEPILOGO FINALE:")
print("="*70)
print(f"- Prodotti trigonometrici distinti: {len([k for k, v in trig_coefficients.items() if v != 0])}")
for k in range(len(sorted_keys)):
    print(f"  • Prodotto #{k}: {sorted_keys[k]}")

print(f"- Analisi completata con successo!")


Analizzando ogni termine dell'espressione...


Analisi termini angolari:   0%|          | 0/18 [00:00<?, ?it/s]


Trovati 2 prodotti trigonometrici distinti
Ordinando i prodotti trigonometrici per complessità

RIEPILOGO FINALE:
- Prodotti trigonometrici distinti: 2
  • Prodotto #0: COSTANTE
  • Prodotto #1: cos(thetal)**2
- Analisi completata con successo!


In [ ]:
# Sostituzioni per esprimere i fattori di forma in termini dei loro moduli quadrati
for key in sorted_keys:
    coeff = trig_coefficients[key]
    for x in [FV,FA,FP]:
        coeff = coeff.subs(x*sp.conjugate(x), sp.Abs(x)**2)
        coeff = coeff.subs(sp.conjugate(x)*x, sp.Abs(x)**2)
    trig_coefficients[key] = coeff

In [23]:
for i, key in enumerate(sorted_keys):
    f = trig_coefficients[key]
    print(f"\n{'-'*50}\nProdotto #{i}: {key}\nCoefficiente:\n")
    sp.pprint(f.collect(sp.Abs(FV)).collect(sp.Abs(FA)).collect(sp.Abs(FP)).collect(sp.Abs(FP)**2).collect(FA*sp.conjugate(FP)).collect(FP*sp.conjugate(FA)))


--------------------------------------------------
Prodotto #0: COSTANTE
Coefficiente:

                                                                          2              ⎛ 2       4   2       2   2   ↪
   ⎛  2        2          2⎞ __      ⎛  2        2          2⎞ __   λ⋅│FV│     2     2   ⎜b ⋅λ   mB ⋅ml    2⋅mB ⋅mK ⋅m ↪
FA⋅⎝mB ⋅ml - mK ⋅ml + ml⋅q ⎠⋅FP + FP⋅⎝mB ⋅ml - mK ⋅ml + ml⋅q ⎠⋅FA + ─────── + q ⋅│FP│  + ⎜──── + ─────── - ─────────── ↪
                                                                       4                 ⎜ 4        2            2     ↪
                                                                                         ⎝         q            q      ↪

↪  2                 4   2                     ⎞      
↪ l        2   2   mK ⋅ml        2   2     2  2⎟     2
↪ ── + 2⋅mB ⋅ml  + ─────── - 2⋅mK ⋅ml  + ml ⋅q ⎟⋅│FA│ 
↪                     2                        ⎟      
↪                    q                         ⎠      

-----------------------------